In [ ]:

# Q32: Comparative Study of Decision Tree, KNN and Naive Bayes

# 1. Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 2. Load dataset
df = pd.read_csv("Student_Performance(2).csv")

print("First five rows:")
print(df.head())

# 3. Create Pass/Fail target
df["Pass_Fail"] = (df["Performance Index"] >= 40).astype(int)

# 4. Select input features and target
X = df[
    [
        "Hours Studied",
        "Previous Scores",
        "Extracurricular Activities",
        "Sleep Hours",
        "Sample Question Papers Practiced"
    ]
]

y = df["Pass_Fail"]

# 5. Identify numerical and categorical columns
numeric_features = [
    "Hours Studied",
    "Previous Scores",
    "Sleep Hours",
    "Sample Question Papers Practiced"
]

categorical_features = ["Extracurricular Activities"]

# 6. Create preprocessing pipeline
# Missing numerical values -> median
# Missing categorical values -> most frequent
# Categorical values -> One-Hot Encoding

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# 7. Common stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 8. Define three classifiers

# Decision Tree
decision_tree = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# KNN (scaling is important for KNN)
knn = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])

# Naive Bayes
naive_bayes = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GaussianNB())
])

models = {
    "Decision Tree": decision_tree,
    "KNN": knn,
    "Naive Bayes": naive_bayes
}

# 9. Train and evaluate all models
results = []

for name, model in models.items():

    # Train model
    model.fit(X_train, y_train)

    # Predict test data
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    results.append({
        "Model": name,
        "Accuracy": round(accuracy, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-Score": round(f1, 4)
    })

    # Display confusion matrix
    print("\n", "=" * 40)
    print(name)
    print("=" * 40)

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

# 10. Display comparison table
comparison = pd.DataFrame(results)

print("\n\nFINAL MODEL COMPARISON")
print(comparison.to_string(index=False))

# 11. Recommend the model based on F1-score
best_model = comparison.loc[
    comparison["F1-Score"].idxmax()
]

print("\nRecommended Model:", best_model["Model"])
print("Best F1-Score:", best_model["F1-Score"])
